# Seismic Geometry Analysis

This notebook processes seismic catalog data by:
1. Loading catalog and station data
2. Adding station coordinates to the catalog
3. Calculating back azimuth and angle of incidence
4. Filtering events based on angle of incidence criteria

## 1. Import Required Libraries

In [33]:
import pandas as pd
import numpy as np
import obspy
import matplotlib.pyplot as plt
import sys
import os

# Add the scripts directory to path to import custom modules
scripts_path = os.path.join('..', 'scripts')
if scripts_path not in sys.path:
    sys.path.append(scripts_path)

# Import seismic geometry functions
try:
    from seismic_geometry import calculate_back_azimuth, calculate_angle_of_incidence, calculate_epicentral_distance_km
    print("Successfully imported seismic geometry functions")
except ImportError as e:
    print(f"Warning: Could not import seismic geometry functions: {e}")
    print("Will define basic geometry functions locally")

Successfully imported seismic geometry functions


## 2. Load Catalog and Station Data

In [ ]:
# Load the filtered catalog sample (100 earthquake events)
#catalog = pd.read_csv('../data/catalog_filtered_sample.csv')
catalog = pd.read_csv('../data/.csv')
print(f"Loaded catalog with {len(catalog)} rows and {len(catalog.columns)} columns")
print(f"Catalog columns: {catalog.columns.tolist()}")
print(f"Unique stations in catalog: {catalog['station'].unique()}")

# Load station information
stations = pd.read_csv('../data/axial_seamount_stations.csv')
print(f"\nLoaded station data with {len(stations)} rows and {len(stations.columns)} columns")
print(f"Station columns: {stations.columns.tolist()}")

# Display first few rows of each dataframe
print("\nFirst 5 rows of catalog:")
display(catalog.head())

print("\nFirst 5 rows of station data:")
display(stations.head())

Loaded catalog with 100 rows and 16 columns
Catalog columns: ['id', 'year', 'datetime', 'lat', 'lon', 'depth', 'mag', 'station', 'time_diff_seconds', 'total_picks', 'p_arrival_time', 'p_weight', 'p_quality', 's_arrival_time', 's_weight', 's_quality']
Unique stations in catalog: ['OOAXEC3' 'OOAXEC2' 'OOAXAS1' 'OOAXEC1' 'OOAXID1' 'OOAXAS2' 'OOAXCC1']

Loaded station data with 8 rows and 5 columns
Station columns: ['Station ID', 'Location Name', 'Latitude (°N)', 'Longitude (°W)', 'Elevation (m)']

First 5 rows of catalog:


,id,year,datetime,lat,lon,depth,mag,station,time_diff_seconds,total_picks,p_arrival_time,p_weight,p_quality,s_arrival_time,s_weight,s_quality
0,1015287,2015,2015-01-01 00:03:39.175,45.93845,-129.96648,0.099,1.0,OOAXEC3,0.085,2,0.58,1.0,EH,1.35,0.5,EH
1,1015291,2015,2015-01-01 01:30:03.214,45.92372,-129.98706,1.109,0.3,OOAXEC3,0.004,2,0.52,1.0,EH,1.03,1.0,EH
2,1015298,2015,2015-01-01 02:54:51.020,45.98200,-130.01392,0.958,0.6,OOAXEC3,0.160,2,1.48,0.5,EH,2.82,1.0,EH
3,1015298,2015,2015-01-01 02:54:51.020,45.98200,-130.01392,0.958,0.6,OOAXEC2,0.160,2,0.73,1.0,HH,1.62,0.2,HH
4,1015315,2015,2015-01-01 03:17:23.950,45.98335,-130.02806,1.383,0.5,OOAXAS1,0.410,2,1.81,-0.5,EH,3.44,-0.5,EH



First 5 rows of station data:


,Station ID,Location Name,Latitude (°N),Longitude (°W),Elevation (m)
0,AXAS1,Axial Ashes 1,45.936,-130.013,-1516
1,AXAS2,Axial Ashes 2,45.936,-130.013,-1516
2,AXBA1,Axial Base 1,45.550,-130.000,-2600
3,AXCC1,Axial Central Caldera 1,45.955,-130.008,-1516
4,AXEC1,Axial East Caldera 1,45.955,-129.973,-1519


## 3. Merge Station Information with Catalog

In [35]:
# Create a station ID mapping - remove 'OO' prefix from catalog station names
catalog['station_id'] = catalog['station'].str.replace('OO', '', regex=False)
print(f"Original station names: {catalog['station'].unique()}")
print(f"Modified station IDs: {catalog['station_id'].unique()}")

Original station names: ['OOAXEC3' 'OOAXEC2' 'OOAXAS1' 'OOAXEC1' 'OOAXID1' 'OOAXAS2' 'OOAXCC1']
Modified station IDs: ['AXEC3' 'AXEC2' 'AXAS1' 'AXEC1' 'AXID1' 'AXAS2' 'AXCC1']


In [36]:
catalog = catalog.drop('station', axis=1)

In [37]:
display(catalog)

,id,year,datetime,lat,lon,depth,mag,time_diff_seconds,total_picks,p_arrival_time,p_weight,p_quality,s_arrival_time,s_weight,s_quality,station_id
0,1015287,2015,2015-01-01 00:03:39.175,45.93845,-129.96648,0.099,1.0,0.085,2,0.58,1.0,EH,1.35,0.5,EH,AXEC3
1,1015291,2015,2015-01-01 01:30:03.214,45.92372,-129.98706,1.109,0.3,0.004,2,0.52,1.0,EH,1.03,1.0,EH,AXEC3
2,1015298,2015,2015-01-01 02:54:51.020,45.98200,-130.01392,0.958,0.6,0.160,2,1.48,0.5,EH,2.82,1.0,EH,AXEC3
3,1015298,2015,2015-01-01 02:54:51.020,45.98200,-130.01392,0.958,0.6,0.160,2,0.73,1.0,HH,1.62,0.2,HH,AXEC2
4,1015315,2015,2015-01-01 03:17:23.950,45.98335,-130.02806,1.383,0.5,0.410,2,1.81,-0.5,EH,3.44,-0.5,EH,AXAS1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1015490,2015,2015-01-01 22:09:40.707,45.94970,-130.02407,0.548,0.5,0.063,2,0.91,1.0,EH,1.76,0.5,EH,AXAS1
96,1015490,2015,2015-01-01 22:09:40.707,45.94970,-130.02407,0.548,0.5,0.063,2,1.80,0.5,HH,2.57,0.5,HH,AXCC1
97,1015495,2015,2015-01-01 22:10:59.640,45.94110,-130.01340,0.516,0.1,0.110,2,0.76,1.0,EH,1.56,-0.5,EH,AXEC3
98,1015499,2015,2015-01-01 22:11:33.273,45.95124,-129.95463,0.108,0.1,0.637,2,1.28,1.0,HH,2.02,0.2,HH,AXCC1


In [16]:
# Create new columns 'station_lat', 'station_lon' and 'station_elev' in df catalog.
# For each row in df catalog, where 'station_id' = 'Station_ID' in df stations, add 'Latitude (N)' row value in df stations as row value in 'station_lat' in df catalog
# Repeat for 'Longitude (W)' in 'station_lon', and 'Elevation (m)' in 'station_elev'

# First, let's check the exact column names in the stations dataframe
print("Station dataframe columns:")
print(stations.columns.tolist())

# Merge catalog with station information
# Map station_id in catalog to Station_ID in stations, and add the coordinate columns
catalog_with_stations = catalog.merge(
    stations[['Station ID', 'Latitude (°N)', 'Longitude (°W)', 'Elevation (m)']], 
    left_on='station_id', 
    right_on='Station ID', 
    how='left'
)

# Rename the station coordinate columns to more convenient names
catalog_with_stations = catalog_with_stations.rename(columns={
    'Latitude (°N)': 'station_lat',
    'Longitude (°W)': 'station_lon', 
    'Elevation (m)': 'station_elev'
})

# Drop the duplicate Station_ID column (we already have station_id)
catalog_with_stations = catalog_with_stations.drop('Station ID', axis=1)

print(f"\nMerged catalog shape: {catalog_with_stations.shape}")
print(f"Number of rows with station coordinates: {catalog_with_stations['station_lat'].notna().sum()}")

# Display the merged dataframe
print("\nFirst 5 rows of merged catalog with station coordinates:")
display(catalog_with_stations.head())

# Check for any missing station matches
missing_stations = catalog_with_stations['station_lat'].isna().sum()
if missing_stations > 0:
    print(f"\nWarning: {missing_stations} rows have missing station coordinates")
    print("Station IDs without matches:")
    missing_ids = catalog_with_stations[catalog_with_stations['station_lat'].isna()]['station_id'].unique()
    print(missing_ids)

Station dataframe columns:
['Station ID', 'Location Name', 'Latitude (°N)', 'Longitude (°W)', 'Elevation (m)']

Merged catalog shape: (100, 19)
Number of rows with station coordinates: 100

First 5 rows of merged catalog with station coordinates:


,id,year,datetime,lat,lon,depth,mag,time_diff_seconds,total_picks,p_arrival_time,p_weight,p_quality,s_arrival_time,s_weight,s_quality,station_id,station_lat,station_lon,station_elev
0,1015287,2015,2015-01-01 00:03:39.175,45.93845,-129.96648,0.099,1.0,0.085,2,0.58,1.0,EH,1.35,0.5,EH,AXEC3,45.955,-129.973,-1519
1,1015291,2015,2015-01-01 01:30:03.214,45.92372,-129.98706,1.109,0.3,0.004,2,0.52,1.0,EH,1.03,1.0,EH,AXEC3,45.955,-129.973,-1519
2,1015298,2015,2015-01-01 02:54:51.020,45.98200,-130.01392,0.958,0.6,0.160,2,1.48,0.5,EH,2.82,1.0,EH,AXEC3,45.955,-129.973,-1519
3,1015298,2015,2015-01-01 02:54:51.020,45.98200,-130.01392,0.958,0.6,0.160,2,0.73,1.0,HH,1.62,0.2,HH,AXEC2,45.955,-129.973,-1519
4,1015315,2015,2015-01-01 03:17:23.950,45.98335,-130.02806,1.383,0.5,0.410,2,1.81,-0.5,EH,3.44,-0.5,EH,AXAS1,45.936,-130.013,-1516


In [17]:
# Convert station elevation from meters to kilometers
catalog_with_stations['station_elev'] = catalog_with_stations['station_elev'] / 1000.0

print("Station elevations converted from meters to kilometers:")
print(f"New range: {catalog_with_stations['station_elev'].min():.3f} km to {catalog_with_stations['station_elev'].max():.3f} km")

# Display a sample of the converted data
display(catalog_with_stations[['station_id', 'station_lat', 'station_lon', 'station_elev']].head())

Station elevations converted from meters to kilometers:
New range: -1.519 km to -1.512 km


,station_id,station_lat,station_lon,station_elev
0,AXEC3,45.955,-129.973,-1.519
1,AXEC3,45.955,-129.973,-1.519
2,AXEC3,45.955,-129.973,-1.519
3,AXEC2,45.955,-129.973,-1.519
4,AXAS1,45.936,-130.013,-1.516


## 4. Calculate Seismic Geometry Parameters

In [23]:
# Calculate geometry parameters for each row using imported functions
print("Calculating back azimuth and angle of incidence...")

# Filter out rows without station coordinates
valid_rows = catalog_with_stations['station_lat'].notna()
print(f"Rows with valid station coordinates: {valid_rows.sum()}")

# Calculate back azimuth, epicentral distance, and angle of incidence
catalog_with_stations['back_azimuth'] = np.nan
catalog_with_stations['epicentral_distance_km'] = np.nan
catalog_with_stations['angle_of_incidence'] = np.nan

for idx, row in catalog_with_stations[valid_rows].iterrows():
    try:
        # Calculate back azimuth (earthquake to station)
        baz = calculate_back_azimuth(
            row['lat'], row['lon'], 
            row['station_lat'], row['station_lon']
        )
        catalog_with_stations.loc[idx, 'back_azimuth'] = baz
        
        # Calculate epicentral distance in km
        epicentral_dist = calculate_epicentral_distance_km(
            row['lat'], row['lon'], 
            row['station_lat'], row['station_lon']
        )
        catalog_with_stations.loc[idx, 'epicentral_distance_km'] = epicentral_dist
        
        # Calculate angle of incidence using epicentral distance
        aoi = calculate_angle_of_incidence(
            epicentral_dist, row['depth'], row['station_elev']
        )
        catalog_with_stations.loc[idx, 'angle_of_incidence'] = aoi
        
    except Exception as e:
        print(f"Error calculating geometry for row {idx}: {e}")

print(f"Successfully calculated geometry for {catalog_with_stations['back_azimuth'].notna().sum()} rows")

# Display statistics
print(f"\nBack azimuth range: {catalog_with_stations['back_azimuth'].min():.1f} to {catalog_with_stations['back_azimuth'].max():.1f} degrees")
print(f"Epicentral distance range: {catalog_with_stations['epicentral_distance_km'].min():.1f} to {catalog_with_stations['epicentral_distance_km'].max():.1f} km")
print(f"Angle of incidence range: {catalog_with_stations['angle_of_incidence'].min():.1f} to {catalog_with_stations['angle_of_incidence'].max():.1f} degrees")

# Show sample results
print("\nSample of calculated geometry parameters:")
display(catalog_with_stations[['station_id', 'lat', 'lon', 'depth', 'station_lat', 'station_lon', 'epicentral_distance_km', 'back_azimuth', 'angle_of_incidence']].head())

Calculating back azimuth and angle of incidence...
Rows with valid station coordinates: 100
Successfully calculated geometry for 100 rows

Back azimuth range: 0.2 to 359.6 degrees
Epicentral distance range: 0.5 to 5.4 km
Angle of incidence range: 11.1 to 68.6 degrees

Sample of calculated geometry parameters:


,station_id,lat,lon,depth,station_lat,station_lon,epicentral_distance_km,back_azimuth,angle_of_incidence
0,AXEC3,45.93845,-129.96648,0.099,45.955,-129.973,1.908072,164.678449,49.702841
1,AXEC3,45.92372,-129.98706,1.109,45.955,-129.973,3.644141,197.363408,54.202454
2,AXEC3,45.98200,-130.01392,0.958,45.955,-129.973,4.360663,313.525232,60.401998
3,AXEC2,45.98200,-130.01392,0.958,45.955,-129.973,4.360663,313.525232,60.401998
4,AXAS1,45.98335,-130.02806,1.383,45.936,-130.013,5.392239,347.537786,61.736433


## 5. Filter by Angle of Incidence

In [25]:
# Filter earthquakes with angle of incidence <= 30 degrees
print(f"Original dataset shape: {catalog_with_stations.shape}")
print(f"Rows with valid angle of incidence: {catalog_with_stations['angle_of_incidence'].notna().sum()}")

# Create filter for angle of incidence <= 30 degrees
aoi_filter = catalog_with_stations['angle_of_incidence'] <= 30.0
valid_aoi = catalog_with_stations['angle_of_incidence'].notna()

# Apply both filters (valid and <= 30 degrees)
final_filter = valid_aoi & aoi_filter

catalog_final = catalog_with_stations[final_filter].copy()

print(f"\nFiltered dataset shape: {catalog_final.shape}")
print(f"Rows removed: {catalog_with_stations.shape[0] - catalog_final.shape[0]}")

# Statistics about filtering
print(f"\nAngle of incidence statistics before filtering:")
print(f"  Mean: {catalog_with_stations['angle_of_incidence'].mean():.1f} degrees")
print(f"  Min: {catalog_with_stations['angle_of_incidence'].min():.1f} degrees") 
print(f"  Max: {catalog_with_stations['angle_of_incidence'].max():.1f} degrees")
print(f"  Rows > 30 degrees: {(catalog_with_stations['angle_of_incidence'] > 30).sum()}")

print(f"\nAngle of incidence statistics after filtering:")
print(f"  Mean: {catalog_final['angle_of_incidence'].mean():.1f} degrees")
print(f"  Min: {catalog_final['angle_of_incidence'].min():.1f} degrees")
print(f"  Max: {catalog_final['angle_of_incidence'].max():.1f} degrees")

# Display distribution by station
print(f"\nEvent distribution by station after filtering:")
print(catalog_final['station_id'].value_counts().sort_index())

# Show final dataframe
print(f"\nFinal filtered catalog:")
display(catalog_final.head())

Original dataset shape: (100, 22)
Rows with valid angle of incidence: 100

Filtered dataset shape: (13, 22)
Rows removed: 87

Angle of incidence statistics before filtering:
  Mean: 47.3 degrees
  Min: 11.1 degrees
  Max: 68.6 degrees
  Rows > 30 degrees: 87

Angle of incidence statistics after filtering:
  Mean: 21.4 degrees
  Min: 11.1 degrees
  Max: 29.0 degrees

Event distribution by station after filtering:
station_id
AXAS1    7
AXEC2    1
AXEC3    4
AXID1    1
Name: count, dtype: int64

Final filtered catalog:


,id,year,datetime,lat,lon,depth,mag,time_diff_seconds,total_picks,p_arrival_time,...,s_arrival_time,s_weight,s_quality,station_id,station_lat,station_lon,station_elev,back_azimuth,angle_of_incidence,epicentral_distance_km
11,1015330,2015,2015-01-01 05:14:20.784,45.93734,-130.02082,1.205,0.8,0.016,2,0.57,...,1.21,1.0,EH,AXAS1,45.936,-130.013,-1.516,283.844437,12.892419,0.622813
12,1015330,2015,2015-01-01 05:14:20.784,45.93734,-130.02082,1.205,0.8,0.016,2,0.79,...,1.50,0.5,EH,AXID1,45.934,-130.004,-1.512,285.941436,26.467297,1.352710
17,1015346,2015,2015-01-01 09:15:46.225,45.94279,-130.01436,0.501,0.1,0.105,2,0.71,...,1.39,0.5,EH,AXAS1,45.936,-130.013,-1.516,352.070858,20.703522,0.762302
20,1015350,2015,2015-01-01 09:47:00.989,45.94051,-130.01304,0.462,0.1,0.149,2,0.69,...,1.33,0.5,EH,AXAS1,45.936,-130.013,-1.516,359.646623,14.226886,0.501499
23,1015355,2015,2015-01-01 10:17:29.103,45.96533,-129.98912,3.211,0.1,0.013,2,1.04,...,1.95,-0.5,EH,AXEC3,45.955,-129.973,-1.519,312.676632,19.711976,1.694701


## 6. Save Filtered Results

In [27]:
# Save the final filtered catalog with geometry parameters
output_file = '../data/catalog_with_geometry_filtered.csv'
catalog_final.to_csv(output_file, index=False)

print(f"Saved filtered catalog with geometry parameters to: {output_file}")
print(f"Final dataset contains {len(catalog_final)} rows with {len(catalog_final.columns)} columns")

print(f"\nFinal column list:")
for i, col in enumerate(catalog_final.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\nSummary statistics:")
print(f"  Total events: {len(catalog_final)}")
print(f"  Unique earthquake IDs: {catalog_final['id'].nunique()}")
print(f"  Stations represented: {catalog_final['station_id'].nunique()}")
print(f"  Year range: {catalog_final['year'].min()} - {catalog_final['year'].max()}")
print(f"  Magnitude range: {catalog_final['mag'].min():.1f} - {catalog_final['mag'].max():.1f}")
print(f"  Depth range: {catalog_final['depth'].min():.1f} - {catalog_final['depth'].max():.1f} km")
print(f"  Back azimuth range: {catalog_final['back_azimuth'].min():.1f} - {catalog_final['back_azimuth'].max():.1f} degrees")
print(f"  Angle of incidence range: {catalog_final['angle_of_incidence'].min():.1f} - {catalog_final['angle_of_incidence'].max():.1f} degrees")

print(f"\nDataset is ready for shear-wave splitting analysis!")

Saved filtered catalog with geometry parameters to: ../data/catalog_with_geometry_filtered.csv
Final dataset contains 13 rows with 22 columns

Final column list:
   1. id
   2. year
   3. datetime
   4. lat
   5. lon
   6. depth
   7. mag
   8. time_diff_seconds
   9. total_picks
  10. p_arrival_time
  11. p_weight
  12. p_quality
  13. s_arrival_time
  14. s_weight
  15. s_quality
  16. station_id
  17. station_lat
  18. station_lon
  19. station_elev
  20. back_azimuth
  21. angle_of_incidence
  22. epicentral_distance_km

Summary statistics:
  Total events: 13
  Unique earthquake IDs: 11
  Stations represented: 4
  Year range: 2015 - 2015
  Magnitude range: -9.0 - 0.8
  Depth range: 0.3 - 3.2 km
  Back azimuth range: 0.2 - 359.6 degrees
  Angle of incidence range: 11.1 - 29.0 degrees

Dataset is ready for shear-wave splitting analysis!
